In [1]:
import pandas as pd

# 数据文件路径
data_file = 'raw_data/2.5_all_data.csv'

# 加载数据
data = pd.read_csv(data_file)

In [2]:
from PIL import Image
import io

image_size = (150, 150)

def hex_to_image(hex_data):
    if pd.isna(hex_data):
        # 如果数据缺失，生成空白图像
        blank_image = Image.new('L', image_size, 255)  # 'L' 模式表示灰度图像，0 表示黑色
        return blank_image
    image_data = bytes.fromhex(hex_data[2:])  # 移除前缀'0x'
    image = Image.open(io.BytesIO(image_data))
    image = image.resize(image_size)  # 调整图像大小
    return image

In [3]:
import io
import os
def process_and_save_data(data, root_dir='dataset'):
    categories = data['result'].unique()  # 获取所有独特的分类结果
    for category in categories:
        category_dir = os.path.join(root_dir, category)
        os.makedirs(category_dir, exist_ok=True)  # 为每个分类创建文件夹

    for index, row in data.iterrows():
        patient_id = row['patient ID']
        patient_dir = os.path.join(root_dir, row['result'], str(patient_id))
        os.makedirs(patient_dir, exist_ok=True)  # 为每个患者创建文件夹

        # 拼接所有图像
        images = []
        for tx_col in ['TX0', 'TX1', 'TX2', 'TX3', 'TX4', 'TX5', 'TX6']:
            hex_data = row.get(tx_col)
            image = hex_to_image(hex_data)
            images.append(image)
        
        # 横向拼接图片
        combined_image = Image.new('L', (image_size[0] * len(images), image_size[1]))
        for i, img in enumerate(images):
            combined_image.paste(img, (i * image_size[0], 0))
        
        combined_image_path = os.path.join(patient_dir, 'combined_image.jpg')
        combined_image.save(combined_image_path)

        # 定义需要保存的数值特征
        feature_columns = ['Gender', 'Age', 'ALB', 'ALT', 'AST', 'GLO', 'WBC', 'TP']
        features = row[feature_columns]
        
        # 保存数值特征到CSV
        features.to_frame().T.to_csv(os.path.join(patient_dir, 'features.csv'), index=False)

process_and_save_data(data)

In [4]:
import os
from PIL import Image

def check_image_sizes(root_dir='dataset', expected_size=(150 * 7, 150)):
    all_images_same_size = True
    mismatched_images = []
    
    # 遍历数据集目录，检查所有图像文件
    for category in os.listdir(root_dir):
        category_path = os.path.join(root_dir, category)
        if os.path.isdir(category_path):
            for patient_id in os.listdir(category_path):
                patient_path = os.path.join(category_path, patient_id)
                if os.path.isdir(patient_path):
                    image_path = os.path.join(patient_path, 'combined_image.jpg')
                    if os.path.exists(image_path):
                        with Image.open(image_path) as img:
                            if img.size != expected_size:
                                all_images_same_size = False
                                mismatched_images.append(image_path)
    
    if all_images_same_size:
        print("所有图像尺寸均一致。")
    else:
        print(f"以下图像尺寸不一致（期望尺寸为 {expected_size}）：")
        for image_path in mismatched_images:
            print(image_path)

# 运行检查代码
check_image_sizes()

所有图像尺寸均一致。
